Start new

**NEW Results**

Block 1: Complete Training, All Visualizations (A-to-Z) & Dynamic Diagram

In [1]:
!pip install timm ttach grad-cam scikit-learn shap captum torchinfo seaborn graphviz
!apt-get install -y graphviz

import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torch.amp import GradScaler, autocast
from torchinfo import summary
from graphviz import Digraph

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, roc_curve, auc, 
                             precision_recall_curve, average_precision_score, 
                             classification_report)
from sklearn.manifold import TSNE

# XAI Imports
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import shap
from captum.attr import IntegratedGradients
from captum.attr import visualization as viz

print("🚀 Starting Complete GreenXAI-PCOS Pipeline (Leakage-Free & Teacher Pre-trained)...")

# ==========================================
# 1. Configuration & Separate Output Folder
# ==========================================
OUTPUT_FOLDER = '/kaggle/working/GreenXAI_Paper_Outputs/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

CONFIG = {
    'data_dir': '/kaggle/input/datasets/shuvokarmaker/pcos-12k/PCOS', 
    'output_dir': OUTPUT_FOLDER, 
    'batch_size': 32, 
    'teacher_epochs': 5,   # Teacher learns the task first
    'student_epochs': 20,  # Student distills knowledge over 20 epochs
    'learning_rate': 0.001,
    'temperature': 3.0, 
    'alpha': 0.5,       
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

# ==========================================
# 2. Data Loading (Isolated Transforms - No Leakage)
# ==========================================
def get_dataloaders():
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    dataset_train = datasets.ImageFolder(root=CONFIG['data_dir'], transform=train_transforms)
    dataset_test = datasets.ImageFolder(root=CONFIG['data_dir'], transform=test_transforms)
    
    torch.manual_seed(42)
    indices = torch.randperm(len(dataset_train)).tolist()
    train_size = int(0.8 * len(dataset_train))
    
    train_subset = Subset(dataset_train, indices[:train_size])
    test_subset = Subset(dataset_test, indices[train_size:])

    train_loader = DataLoader(train_subset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_subset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, test_loader, dataset_train.classes

# ==========================================
# 3. Model Architecture & Loss
# ==========================================
def build_models(num_classes):
    teacher = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    teacher.classifier = nn.Linear(teacher.classifier.in_features, num_classes)
    teacher = teacher.to(CONFIG['device'])
    
    student = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    student.classifier[3] = nn.Linear(student.classifier[3].in_features, num_classes)
    student = student.to(CONFIG['device'])
    return teacher, student

def distillation_loss(student_logits, teacher_logits, labels):
    T, alpha = CONFIG['temperature'], CONFIG['alpha']
    soft_targets = F.softmax(teacher_logits / T, dim=-1)
    student_log_softmax = F.log_softmax(student_logits / T, dim=-1)
    loss_kd = F.kl_div(student_log_softmax, soft_targets, reduction='batchmean') * (T * T)
    loss_ce = F.cross_entropy(student_logits, labels)
    return (alpha * loss_ce) + ((1 - alpha) * loss_kd)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

# ==========================================
# 4. Architecture Diagram Generator (Dynamic Accuracy)
# ==========================================
def generate_architecture_diagram(final_acc_percent):
    dot = Digraph(name="Green-XAI_Framework", format="png")
    dot.attr(rankdir='TB', size='10,10', dpi='300')

    dot.node('Data', 'PCOS Ultrasound Dataset\n(12,680 Raw / 11,784 Validated)', shape='cylinder', style='filled', fillcolor='#D6EAF8')
    dot.node('Pre', 'Data Preprocessing\n(Resize, Augmentation, Normalize)', shape='box', style='rounded')
    dot.node('Teacher', 'Teacher Model\n(DenseNet121 - Pre-trained)', shape='box', style='filled', fillcolor='#EAEDED')
    dot.node('Student', 'Student Model\n(MobileNetV3-Small - Lightweight)', shape='box', style='filled', fillcolor='#D5F5E3')
    dot.node('Loss', 'Knowledge Distillation\n(KD Loss + CE Loss)', shape='ellipse', style='filled', fillcolor='#FADBD8')
    dot.node('Eval', 'Dual Validation\n(80-20 Split & 10-Fold CV)', shape='box', style='filled', fillcolor='#FCF3CF')
    dot.node('Metrics', f'Performance Metrics\n(Acc: {final_acc_percent:.2f}%, Precision, F1, ROC)', shape='folder')
    dot.node('XAI', 'Triple-Layer Explainable AI (XAI)\n(Grad-CAM, SHAP, Integrated Gradients)', shape='folder', style='filled', fillcolor='#FDEBD0')

    dot.edge('Data', 'Pre')
    dot.edge('Pre', 'Teacher', label=' Train Data')
    dot.edge('Pre', 'Student', label=' Train Data')
    dot.edge('Teacher', 'Loss', label=' Soft Targets')
    dot.edge('Pre', 'Loss', label=' Hard Targets')
    dot.edge('Loss', 'Student', label=' Optimization')
    dot.edge('Student', 'Eval', label=' Clean Test Data')
    dot.edge('Eval', 'Metrics')
    dot.edge('Eval', 'XAI')

    diagram_path = os.path.join(CONFIG['output_dir'], 'architecture_diagram')
    dot.render(diagram_path, cleanup=True)
    print("✅ Saved: architecture_diagram.png")

# ==========================================
# 5. ALL VISUALIZATIONS & METRICS GENERATOR
# ==========================================
def generate_all_visualizations(student, test_loader, classes, history, teacher_best_acc):
    print("\n📊 Generating ALL Paper-Ready Visualizations & Metrics...")
    student.eval()
    
    all_preds, all_labels, all_probs, all_features = [], [], [], []
    test_images_sample, test_labels_sample = [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
            if len(test_images_sample) < 30:
                test_images_sample.extend(images)
                test_labels_sample.extend(labels)
                
            features = student.features(images)
            pooled_features = F.adaptive_avg_pool2d(features, (1, 1)).flatten(1)
            outputs = student(images)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_features.extend(pooled_features.cpu().numpy())

    # --- Compute All Exact Metrics ---
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro')
    rec = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')
    specificity = tn / (tn + fp)
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    pr_auc = average_precision_score(all_labels, all_probs)
    cls_report = classification_report(all_labels, all_preds, target_names=classes, digits=4)

    # Save Metrics Summary Text File
    report_text = f"""==================================================
GREENXAI-PCOS: FINAL 80:20 HOLDOUT RESULTS SUMMARY
==================================================
Teacher (DenseNet121) Best Val Acc : {teacher_best_acc*100:.2f}%
Student (MobileNetV3) Best Val Acc : {acc*100:.2f}%
--------------------------------------------------
Precision (Macro) : {prec*100:.2f}%
Recall / Sens.    : {rec*100:.2f}%
Specificity       : {specificity*100:.2f}%
F1-Score (Macro)  : {f1:.4f}
ROC-AUC           : {roc_auc:.4f}
PR-AUC (AP)       : {pr_auc:.4f}
--------------------------------------------------
Confusion Matrix Breakdown:
True Positives (TP - noninfected) : {tp}
True Negatives (TN - infected)    : {tn}
False Positives (FP)              : {fp}
False Negatives (FN)              : {fn}
--------------------------------------------------
Detailed Classification Report:
{cls_report}
==================================================
"""
    print(report_text)
    with open(os.path.join(CONFIG['output_dir'], 'paper_results_summary.txt'), 'w') as f:
        f.write(report_text)

    # 5.1 Learning Curve
    plt.figure(figsize=(8, 5))
    plt.plot(history['train_loss'], label='Student Train Loss', color='red', marker='o')
    plt.plot(history['val_acc'], label='Student Val Accuracy', color='blue', marker='s')
    plt.title('Student Model Convergence: Loss & Accuracy')
    plt.xlabel('Epochs')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(os.path.join(CONFIG['output_dir'], 'learning_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: learning_curve.png")

    # 5.2 Confusion Matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix (Student Model)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: confusion_matrix.png")

    # 5.3 ROC & PR Curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[0].set_title('ROC Curve')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].legend(loc="lower right")

    precision_c, recall_c, _ = precision_recall_curve(all_labels, all_probs)
    axes[1].plot(recall_c, precision_c, color='purple', lw=2, label=f'AP = {pr_auc:.4f}')
    axes[1].set_title('Precision-Recall Curve')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].legend(loc="lower left")
    plt.savefig(os.path.join(CONFIG['output_dir'], 'roc_pr_curves.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: roc_pr_curves.png")

    # 5.4 t-SNE Plot
    print("⏳ Generating t-SNE Plot...")
    tsne = TSNE(n_components=2, random_state=42)
    tsne_res = tsne.fit_transform(np.array(all_features))
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=tsne_res[:, 0], y=tsne_res[:, 1], hue=[classes[l] for l in all_labels], palette=['blue', 'orange'], alpha=0.7)
    plt.title('t-SNE Feature Clusters')
    plt.savefig(os.path.join(CONFIG['output_dir'], 'tsne_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: tsne_plot.png")

    # Utils for XAI
    inv_normalize = transforms.Normalize([-0.485/0.229, -0.456/0.224, -0.406/0.225], [1/0.229, 1/0.224, 1/0.225])
    def format_img(tensor):
        return np.clip(np.transpose(inv_normalize(tensor).cpu().numpy(), (1, 2, 0)), 0, 1)

    # 5.5 Grad-CAM Grid
    print("⏳ Generating Grad-CAM Grid...")
    cam = GradCAM(model=student, target_layers=[student.features[-1]])
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for i in range(4):
        img_t, lbl = test_images_sample[i], test_labels_sample[i].item()
        pred = torch.argmax(student(img_t.unsqueeze(0))).item()
        grayscale_cam = cam(input_tensor=img_t.unsqueeze(0), targets=[ClassifierOutputTarget(pred)])[0, :]
        vis_img = show_cam_on_image(format_img(img_t), grayscale_cam, use_rgb=True)
        axes[0, i].imshow(format_img(img_t)); axes[0, i].set_title(f"True: {classes[lbl]}"); axes[0, i].axis('off')
        axes[1, i].imshow(vis_img); axes[1, i].set_title(f"Pred: {classes[pred]}"); axes[1, i].axis('off')
    plt.savefig(os.path.join(CONFIG['output_dir'], 'gradcam_grid.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: gradcam_grid.png")

    # 5.6 SHAP Analysis
    print("⏳ Generating SHAP Analysis...")
    bg = torch.stack(test_images_sample[:20]).to(CONFIG['device'])
    test_imgs = torch.stack(test_images_sample[20:23]).to(CONFIG['device'])
    explainer = shap.GradientExplainer(student, bg)
    shap_values, _ = explainer.shap_values(test_imgs, ranked_outputs=1)
    shap_numpy = [np.transpose(s, (0, 2, 3, 1)) for s in shap_values]
    test_numpy = np.array([format_img(img) for img in test_imgs])
    plt.figure()
    shap.image_plot(shap_numpy, test_numpy, show=False)
    plt.savefig(os.path.join(CONFIG['output_dir'], 'shap_analysis.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: shap_analysis.png")

    # 5.7 Integrated Gradients
    print("⏳ Generating Integrated Gradients...")
    ig = IntegratedGradients(student)
    single_img = test_imgs[0].unsqueeze(0)
    attr, _ = ig.attribute(single_img, target=test_labels_sample[20].item(), return_convergence_delta=True)
    fig, _ = viz.visualize_image_attr_multiple(
        np.transpose(attr.squeeze(0).cpu().detach().numpy(), (1, 2, 0)),
        format_img(test_imgs[0]),
        methods=["original_image", "blended_heat_map"], signs=["all", "all"],
        titles=["Original", "Integrated Gradients"], use_pyplot=False
    )
    fig.savefig(os.path.join(CONFIG['output_dir'], 'integrated_gradients.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: integrated_gradients.png")

    # 5.8 Model Architecture Summary
    summary_str = summary(student, input_size=(1, 3, 224, 224), verbose=0)
    with open(os.path.join(CONFIG['output_dir'], 'model_architecture.txt'), 'w') as f:
        f.write(str(summary_str))
    print("✅ Saved: model_architecture.txt")

    # 5.9 Generate Architecture Diagram with Updated Accuracy
    generate_architecture_diagram(acc * 100)

# ==========================================
# 6. Main Execution Block
# ==========================================
if __name__ == '__main__':
    if not os.path.exists(CONFIG['data_dir']):
        print(f"❌ Error: Dataset path {CONFIG['data_dir']} not found.")
    else:
        train_loader, test_loader, classes = get_dataloaders()
        teacher, student = build_models(len(classes))
        
        # --- PHASE 1: PRE-TRAIN TEACHER ---
        print(f"\n⏳ Phase 1: Pre-training Teacher Model ({CONFIG['teacher_epochs']} epochs)...")
        t_optimizer = optim.Adam(teacher.parameters(), lr=CONFIG['learning_rate'])
        t_scaler = GradScaler('cuda')
        best_t_acc = 0.0
        
        for epoch in range(CONFIG['teacher_epochs']):
            teacher.train()
            running_loss = 0.0
            for images, labels in train_loader:
                images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
                t_optimizer.zero_grad()
                with autocast('cuda'):
                    outputs = teacher(images)
                    loss = F.cross_entropy(outputs, labels)
                t_scaler.scale(loss).backward()
                t_scaler.step(t_optimizer)
                t_scaler.update()
                running_loss += loss.item()
            
            t_acc = evaluate(teacher, test_loader)
            if t_acc > best_t_acc:
                best_t_acc = t_acc
            print(f"Teacher Epoch [{epoch+1}/{CONFIG['teacher_epochs']}] | Loss: {running_loss/len(train_loader):.4f} | Acc: {t_acc:.4f}")

        # --- PHASE 2: TRAIN STUDENT VIA DISTILLATION ---
        print(f"\n⏳ Phase 2: Training Student Model via Knowledge Distillation ({CONFIG['student_epochs']} epochs)...")
        s_optimizer = optim.Adam(student.parameters(), lr=CONFIG['learning_rate'])
        s_scaler = GradScaler('cuda')
        
        best_acc = 0.0
        history = {'train_loss': [], 'val_acc': []}
        
        for epoch in range(CONFIG['student_epochs']):
            student.train()
            teacher.eval()
            running_loss = 0.0
            
            for images, labels in train_loader:
                images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
                s_optimizer.zero_grad()
                
                with autocast('cuda'):
                    with torch.no_grad():
                        teacher_logits = teacher(images)
                    student_logits = student(images)
                    loss = distillation_loss(student_logits, teacher_logits, labels)
                
                s_scaler.scale(loss).backward()
                s_scaler.step(s_optimizer)
                s_scaler.update()
                running_loss += loss.item()
                
            avg_loss = running_loss / len(train_loader)
            acc = evaluate(student, test_loader)
            
            history['train_loss'].append(avg_loss)
            history['val_acc'].append(acc)
            
            print(f"Student Epoch [{epoch+1}/{CONFIG['student_epochs']}] | KD Loss: {avg_loss:.4f} | Acc: {acc:.4f}")
            
            if acc > best_acc:
                best_acc = acc
                torch.save(student.state_dict(), os.path.join(CONFIG['output_dir'], 'best_pcos_student_model.pth'))
                
        print(f"\n✅ Phase 1 & 2 Complete! Best Student Accuracy: {best_acc*100:.2f}%")
        
        # --- PHASE 3: GENERATE ALL OUTPUTS IN FOLDER ---
        student.load_state_dict(torch.load(os.path.join(CONFIG['output_dir'], 'best_pcos_student_model.pth')))
        generate_all_visualizations(student, test_loader, classes, history, best_t_acc)
        print(f"\n🎉 All visualizations & results saved inside: {CONFIG['output_dir']}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 10.5 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.7-py3-none-any.whl size=53345 sha256=52dddab812da7a634a0b5daef2f18f2e23e45e7a422559309f9bfd1bb1269b55
  Stored in directory: /root/.cache/pip/wheels/05/bd/e2/0c0eecf3c26b6cdb231b5656d53905bddc8090d69619d57e6b
Successfully built grad-cam



graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
🚀 Starting Complete GreenXAI-PCOS Pipeline (Leakage-Free & Teacher Pre-trained)...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 170MB/s]


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 99.6MB/s]


⏳ Phase 1: Pre-training Teacher Model (5 epochs)...


Teacher Epoch [1/5] | Loss: 0.1141 | Acc: 0.9699
Teacher Epoch [2/5] | Loss: 0.0800 | Acc: 0.9703
Teacher Epoch [3/5] | Loss: 0.0777 | Acc: 0.9465
Teacher Epoch [4/5] | Loss: 0.0510 | Acc: 0.9856
Teacher Epoch [5/5] | Loss: 0.0471 | Acc: 0.9801

⏳ Phase 2: Training Student Model via Knowledge Distillation (20 epochs)...
Student Epoch [1/20] | KD Loss: 0.1440 | Acc: 0.9792
Student Epoch [2/20] | KD Loss: 0.0593 | Acc: 0.9860
Student Epoch [3/20] | KD Loss: 0.0504 | Acc: 0.9885
Student Epoch [4/20] | KD Loss: 0.0420 | Acc: 0.9860
Student Epoch [5/20] | KD Loss: 0.0395 | Acc: 0.9835
Student Epoch [6/20] | KD Loss: 0.0331 | Acc: 0.9822
Student Epoch [7/20] | KD Loss: 0.0360 | Acc: 0.9881
Student Epoch [8/20] | KD Loss: 0.0310 | Acc: 0.9856
Student Epoch [9/20] | KD Loss: 0.0333 | Acc: 0.9902
Student Epoch [10/20] | KD Loss: 0.0299 | Acc: 0.9919
Student Epoch [11/20] | KD Loss: 0.0303 | Acc: 0.9907
Student Epoch [12/20] | KD Loss: 0.0321 | Acc: 0.9868
Student Epoch [13/20] | KD Loss: 0.0321

Block 2: 10-Fold Cross-Validation (Saves Results & Boxplot in Same Folder)

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import KFold
import numpy as np
import matplotlib.pyplot as plt
from torch.amp import GradScaler, autocast
from sklearn.metrics import accuracy_score

print("🚀 Starting 10-Fold Cross-Validation (Leakage-Free & Teacher Trained)...")

OUTPUT_FOLDER = '/kaggle/working/GreenXAI_Paper_Outputs/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

CONFIG = {
    'data_dir': '/kaggle/input/datasets/shuvokarmaker/pcos-12k/PCOS',
    'output_dir': OUTPUT_FOLDER,
    'batch_size': 32, 
    'teacher_epochs': 5,  # Teacher pre-trained in every fold
    'student_epochs': 20, # Student trained in every fold
    'learning_rate': 0.001,
    'temperature': 3.0, 
    'alpha': 0.5,       
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

def build_models(num_classes):
    teacher = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    teacher.classifier = nn.Linear(teacher.classifier.in_features, num_classes)
    teacher = teacher.to(CONFIG['device'])
    
    student = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    student.classifier[3] = nn.Linear(student.classifier[3].in_features, num_classes)
    student = student.to(CONFIG['device'])
    return teacher, student

def distillation_loss(student_logits, teacher_logits, labels):
    T, alpha = CONFIG['temperature'], CONFIG['alpha']
    soft_targets = F.softmax(teacher_logits / T, dim=-1)
    student_log_softmax = F.log_softmax(student_logits / T, dim=-1)
    loss_kd = F.kl_div(student_log_softmax, soft_targets, reduction='batchmean') * (T * T)
    loss_ce = F.cross_entropy(student_logits, labels)
    return (alpha * loss_ce) + ((1 - alpha) * loss_kd)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

k_folds = 10
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_results = []

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

if not os.path.exists(CONFIG['data_dir']):
    print("❌ Error: Dataset path not found.")
else:
    dataset_train = datasets.ImageFolder(root=CONFIG['data_dir'], transform=train_transforms)
    dataset_test = datasets.ImageFolder(root=CONFIG['data_dir'], transform=test_transforms)
    num_classes = len(dataset_train.classes)

    print(f"Total images found: {len(dataset_train)}. Splitting into {k_folds} Folds...\n")

    for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset_train)):
        print(f"--- 🔄 Starting Fold {fold + 1}/{k_folds} ---")

        train_sub = Subset(dataset_train, train_ids)
        test_sub = Subset(dataset_test, test_ids)

        trainloader = DataLoader(train_sub, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
        testloader = DataLoader(test_sub, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

        teacher, student = build_models(num_classes)
        
        # 1. Pre-train Teacher in this fold
        t_optimizer = optim.Adam(teacher.parameters(), lr=CONFIG['learning_rate'])
        t_scaler = GradScaler('cuda')
        for t_ep in range(CONFIG['teacher_epochs']):
            teacher.train()
            for images, labels in trainloader:
                images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
                t_optimizer.zero_grad()
                with autocast('cuda'):
                    outputs = teacher(images)
                    loss = F.cross_entropy(outputs, labels)
                t_scaler.scale(loss).backward()
                t_scaler.step(t_optimizer)
                t_scaler.update()

        # 2. Train Student via KD in this fold
        optimizer = optim.Adam(student.parameters(), lr=CONFIG['learning_rate'])
        scaler = GradScaler('cuda')
        best_fold_acc = 0.0

        for epoch in range(CONFIG['student_epochs']):
            student.train()
            teacher.eval()
            for images, labels in trainloader:
                images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
                optimizer.zero_grad()
                with autocast('cuda'):
                    with torch.no_grad():
                        teacher_logits = teacher(images)
                    student_logits = student(images)
                    loss = distillation_loss(student_logits, teacher_logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            acc = evaluate(student, testloader)
            if acc > best_fold_acc:
                best_fold_acc = acc

        print(f"✅ Fold {fold + 1} Completed! Best Student Accuracy: {best_fold_acc*100:.2f}%\n")
        fold_results.append(best_fold_acc)

    mean_acc = np.mean(fold_results) * 100
    std_acc = np.std(fold_results) * 100

    # Format & Save 10-Fold Report
    kfold_report = "\n" + "="*50 + "\n🏆 10-FOLD CROSS-VALIDATION RESULTS 🏆\n" + "="*50 + "\n"
    for i, res in enumerate(fold_results):
        kfold_report += f"Fold {i+1:02d}: \t {res*100:.2f}%\n"
    kfold_report += "-"*50 + f"\n🌟 Final Result for Paper: {mean_acc:.2f}% ± {std_acc:.2f}% 🌟\n" + "="*50 + "\n"
    
    print(kfold_report)
    with open(os.path.join(CONFIG['output_dir'], '10fold_cv_results.txt'), 'w') as f:
        f.write(kfold_report)

    # Save 10-Fold Accuracy Bar Chart
    plt.figure(figsize=(9, 4.5))
    folds_x = [f"Fold {i+1}" for i in range(k_folds)]
    accs_y = [r * 100 for r in fold_results]
    bars = plt.bar(folds_x, accs_y, color='#2E86C1', edgecolor='black', alpha=0.85)
    plt.axhline(mean_acc, color='red', linestyle='--', linewidth=1.5, label=f'Mean Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%')
    plt.ylim([min(accs_y) - 1.5, 100.2])
    plt.ylabel('Accuracy (%)')
    plt.title('10-Fold Cross-Validation Performance Stability')
    plt.legend(loc='lower right')
    plt.grid(axis='y', alpha=0.3)
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f'{yval:.2f}%', ha='center', va='bottom', fontsize=8)
    plt.savefig(os.path.join(CONFIG['output_dir'], '10fold_cv_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Saved: 10fold_cv_plot.png & 10fold_cv_results.txt")

🚀 Starting 10-Fold Cross-Validation (Leakage-Free & Teacher Trained)...
Total images found: 11784. Splitting into 10 Folds...

--- 🔄 Starting Fold 1/10 ---
✅ Fold 1 Completed! Best Student Accuracy: 99.58%

--- 🔄 Starting Fold 2/10 ---
✅ Fold 2 Completed! Best Student Accuracy: 99.92%

--- 🔄 Starting Fold 3/10 ---
✅ Fold 3 Completed! Best Student Accuracy: 99.75%

--- 🔄 Starting Fold 4/10 ---
✅ Fold 4 Completed! Best Student Accuracy: 99.92%

--- 🔄 Starting Fold 5/10 ---
✅ Fold 5 Completed! Best Student Accuracy: 99.41%

--- 🔄 Starting Fold 6/10 ---
✅ Fold 6 Completed! Best Student Accuracy: 99.66%

--- 🔄 Starting Fold 7/10 ---
✅ Fold 7 Completed! Best Student Accuracy: 99.49%

--- 🔄 Starting Fold 8/10 ---
✅ Fold 8 Completed! Best Student Accuracy: 98.47%

--- 🔄 Starting Fold 9/10 ---
✅ Fold 9 Completed! Best Student Accuracy: 99.58%

--- 🔄 Starting Fold 10/10 ---
✅ Fold 10 Completed! Best Student Accuracy: 99.83%


🏆 10-FOLD CROSS-VALIDATION RESULTS 🏆
Fold 01: 	 99.58%
Fold 02: 	 99.9

Block 3: One-Click ZIP Archive

In [3]:
import shutil
import os

zip_filename = '/kaggle/working/GreenXAI_All_Paper_Files'
shutil.make_archive(zip_filename, 'zip', '/kaggle/working/GreenXAI_Paper_Outputs')

print("📦 ZIP file created successfully!")
print("Files included in 'GreenXAI_All_Paper_Files.zip':")
for file in sorted(os.listdir('/kaggle/working/GreenXAI_Paper_Outputs')):
    print("  📄", file)

📦 ZIP file created successfully!
Files included in 'GreenXAI_All_Paper_Files.zip':
  📄 10fold_cv_plot.png
  📄 10fold_cv_results.txt
  📄 architecture_diagram.png
  📄 best_pcos_student_model.pth
  📄 confusion_matrix.png
  📄 gradcam_grid.png
  📄 integrated_gradients.png
  📄 learning_curve.png
  📄 model_architecture.txt
  📄 paper_results_summary.txt
  📄 roc_pr_curves.png
  📄 shap_analysis.png
  📄 tsne_plot.png
